# 12. Miopia: a carteira não depende do horizonte
Validação, usando o Investidor que já está pronto. Requisito F12.

In [1]:
import sys, os, tempfile
_cwd = os.getcwd()
RAIZ = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'tests' else _cwd
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import numpy as np, pandas as pd
from app.agente import Investidor
from app.mercado import RendaFixa, RendaVariavel

**Teste**: o alpha tem que sair igual para T = 5, 10, 20 e 40, comparando todos os pares.

In [2]:
import itertools

In [3]:
rng = np.random.default_rng(7)
ruido = rng.normal(0,0.06,300)
ruido -= ruido.mean()
ret = pd.DataFrame({'data': pd.date_range('2000-01',periods=300,freq='MS').strftime('%Y-%m'),'ibov':0.015+ruido})
mkt = RendaVariavel(ret)
rf = RendaFixa(0.10).retorno_livre_risco()

HORIZONTES = (5, 10, 20, 40)
alphas = {T: Investidor(5,0.96,1.0,T).carteira_otima(mkt,rf,n_scenarios=40_000,seed=1)
          for T in HORIZONTES}
for T, a in alphas.items():
    print(f'alpha*(T={T:>3}): {a}')

alpha*(T=  5): [0.43327271]
alpha*(T= 10): [0.43327271]
alpha*(T= 20): [0.43327271]
alpha*(T= 40): [0.43327271]


In [5]:
# Equacao (41): ||alpha*_T - alpha*_T'|| < eps para TODOS os pares
EPS = 1e-12
for T, Tl in itertools.combinations(HORIZONTES, 2):
    d = float(np.linalg.norm(alphas[T] - alphas[Tl]))
    assert d < EPS, f'T={T} vs T={Tl}: ||.|| = {d:.2e}'
print(f'todos os {len(list(itertools.combinations(HORIZONTES,2)))} pares com norma < {EPS:g}')

todos os 6 pares com norma < 1e-12


In [6]:
assert 0.05 < alphas[5][0] < 0.95